In [2]:
import os

if "COLAB_GPU" in os.environ:
  print("[INFO] Running in Google Colab, installing requirements")
  !pip install -U torch
  !pip install PyMuPDF
  !pip install tqdm
  !pip install sentence-transformers
  !pip install accelerate
  !pip install bitsandbytes
  !pip install flash-attn --no-build-isloation

[INFO] Running in Google Colab, installing requirements
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.6/554.6 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.1/553.1 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 MB 16.8 MB/s eta 0:0

In [4]:
#will download the dataset if already dosent exist
import os
import requests

#get the pdf path
pdf_path="human-nutrition-text.pdf"

#download pdf if it already doesn't exists
if not os.path.exists(pdf_path):
  print("File doesn't exist, downloading...")

  #url of that pdf
  url="https://pressbooks.oer.hawaii.edu/humannutrition2/open/download?type=pdf"

  #local file name to save the downloaded pdf
  filename=pdf_path
  #get the file get request
  response=requests.get(url)

  #check if the request was sucessful
  if response.status_code==200:
    #open the file in write mode
    with open(filename, "wb") as file:
      file.write(response.content)
      print(f"File has been downloaded and saved as {filename}")
  else:
    print(f"Failed to download the file. Status code :{response.status_code}")
else:
  print(f"File {pdf_path} Exists")

File doesn't exist, downloading...
File has been downloaded and saved as human-nutrition-text.pdf


PDF acquired!

We can import the pages of our PDF to text by first defining the PDF path and then opening and reading it with PyMuPDF (`import fitz`).

We'll write a small helper function to preprocess the text as it gets read. Note that not all text will be read in the same so keep this in mind for when you prepare your text.

We'll save each page to a dictionary and then append that dictionary to a list for ease of use later.

In [24]:
import pymupdf
from tqdm.auto import tqdm
def text_formatter(text: str) -> str:
  #performs minor formatting on text
  cleaned_text=text.replace("\n", " ").strip() #can be different for each doc (best to experimnet)

  return cleaned_text

#Open the pdf and read lines/ Pages
#this only focuses on text rather than imgaes/figures etc

def open_and_read_pdf(pdf_path: str) -> list[dict]:
  """
  Opens a pdf and reads its text content page by page, and collects statistics

  Parameters:
    pdf_path (str): the file path of the documnet to be opened and read

  Returns :
  List of dictionaries (list[dict]): each containing the page number, char count, word count, sentence count,
  token_count,  and the extracted text "
  """
  doc = fitz.open(pdf_path) #open the doc
  pages_and_texts=[]
  #iterate doc pages
  for page_numbers, page in tqdm(enumerate(doc)):
    text=page.get_text() #gets plain text encoded UTF-8
    text=text_formatter(text)
    pages_and_texts.append({
        "Page_number": page_numbers -41,
        "page_char_count": len(text),
        "page_word_count": len(text.split(" ")),
        "page_sentence_count": len(text.split(". ")),
        "page_token_count": len(text)/4,
        "text": text
    })
  return pages_and_texts
pages_and_texts=open_and_read_pdf(pdf_path=pdf_path)
pages_and_texts[:2]

0it [00:00, ?it/s]

[{'Page_number': -41,
  'page_char_count': 29,
  'page_word_count': 4,
  'page_sentence_count': 1,
  'page_token_count': 7.25,
  'text': 'Human Nutrition: 2020 Edition'},
 {'Page_number': -40,
  'page_char_count': 0,
  'page_word_count': 1,
  'page_sentence_count': 1,
  'page_token_count': 0.0,
  'text': ''}]

In [25]:
#lets get a random sample of the pages
import random

random.sample(pages_and_texts, k=3)

[{'Page_number': 839,
  'page_char_count': 458,
  'page_word_count': 84,
  'page_sentence_count': 4,
  'page_token_count': 114.5,
  'text': 'Hawai‘i, poi (pounded taro) was a staple food in the diet and  is still popular today due to it’s nutrient-dense structure.  Poi is high in easily digestible calories,\xa0a good source of  potassium along with a number of other essential vitamins  and minerals. Poi also has many gastrointestinal tract  health benefits due to its fiber and probiotic content.\xa0 With  it’s viscous texture, poi is an excellent first food for infants  to consume.  Infancy  |  839'},
 {'Page_number': 312,
  'page_char_count': 748,
  'page_word_count': 122,
  'page_sentence_count': 4,
  'page_token_count': 187.0,
  'text': 'Learning Activities  Technology Note: The second edition of the Human  Nutrition Open Educational Resource (OER) textbook  features interactive learning activities.\xa0 These activities are  available in the web-based textbook and not available in t

### Get some stats on the text

Let's perform a rough exploratory data analysis (EDA) to get an idea of the size of the texts (e.g. character counts, word counts etc) we're working with.

The different sizes of texts will be a good indicator into how we should split our texts.

Many embedding models have limits on the size of texts they can ingest, for example, the [`sentence-transformers`](https://www.sbert.net/docs/pretrained_models.html) model [`all-mpnet-base-v2`](https://huggingface.co/sentence-transformers/all-mpnet-base-v2) has an input size of 384 tokens.

This means that the model has been trained in ingest and turn into embeddings texts with 384 tokens (1 token ~= 4 characters ~= 0.75 words).

Texts over 384 tokens which are encoded by this model will be auotmatically reduced to 384 tokens in length, potentially losing some information.

We'll discuss this more in the embedding section.

For now, let's turn our list of dictionaries into a DataFrame and explore it.

In [26]:
import pandas as pd

df=pd.DataFrame(pages_and_texts)
df.head()

,Page_number,page_char_count,page_word_count,page_sentence_count,page_token_count,text
0,-41,29,4,1,7.25,Human Nutrition: 2020 Edition
1,-40,0,1,1,0.00,
2,-39,320,54,1,80.00,Human Nutrition: 2020 Edition UNIVERSITY OF ...
3,-38,212,32,1,53.00,Human Nutrition: 2020 Edition by University of...
4,-37,797,145,2,199.25,Contents Preface University of Hawai‘i at Mā...


In [27]:
#stats
df.describe().round(2)

,Page_number,page_char_count,page_word_count,page_sentence_count,page_token_count
count,1208.00,1208.00,1208.00,1208.00,1208.00
mean,562.50,1148.00,198.30,9.97,287.00
std,348.86,560.38,95.76,6.19,140.10
min,-41.00,0.00,1.00,1.00,0.00
25%,260.75,762.00,134.00,4.00,190.50
50%,562.50,1231.50,214.50,10.00,307.88
75%,864.25,1603.50,271.00,14.00,400.88
max,1166.00,2308.00,429.00,32.00,577.00


### Further text processing (splitting pages into sentences)

The ideal way of processing text before embedding it is still an active area of research.

A simple method I've found helpful is to break the text into chunks of sentences.

As in, chunk a page of text into groups of 5, 7, 10 or more sentences (these values are not set in stone and can be explored).

But we want to follow the workflow of:

`Ingest text -> split it into groups/chunks -> embed the groups/chunks -> use the embeddings`

Some options for splitting text into sentences:

1. Split into sentences with simple rules (e.g. split on ". " with `text = text.split(". ")`, like we did above).
2. Split into sentences with a natural language processing (NLP) library such as [spaCy](https://spacy.io/) or [nltk](https://www.nltk.org/).

Why split into sentences?

* Easier to handle than larger pages of text (especially if pages are densely filled with text).
* Can get specific and find out which group of sentences were used to help within a RAG pipeline.

> **Resource:** See [spaCy install instructions](https://spacy.io/usage).

Let's use spaCy to break our text into sentences since it's likely a bit more robust than just using `text.split(". ")`.

In [28]:
from spacy.lang.en import English

nlp=English()

#add a sentencizer pipeline
nlp.add_pipe("sentencizer")

#create a doc instance as example
doc=nlp("This is a sentence. This is another sentence.")

assert len(list(doc.sents))==2

#access the sentence of the content
list(doc.sents)



[This is a sentence., This is another sentence.]

In [29]:
#lets run our own small sentencizing pipeline on our pages of text

for item in tqdm(pages_and_texts):
  item["sentences"]= list(nlp(item["text"]).sents)

  #make sure all sentences are string not objects
  item["sentences"]= [str(sentence) for sentence in item["sentences"]]

  #count the sentences
  item["page_sentence_count_spacy"]=len(item["sentences"])

  0%|          | 0/1208 [00:00<?, ?it/s]

In [30]:
#lets inspect a random sample
random.sample(pages_and_texts, k=3)

[{'Page_number': 1121,
  'page_char_count': 1844,
  'page_word_count': 302,
  'page_sentence_count': 12,
  'page_token_count': 461.0,
  'text': 'Undernutrition,  Overnutrition, and  Malnutrition  UNIVERSITY OF HAWAI‘I AT MĀNOA FOOD SCIENCE AND HUMAN  NUTRITION PROGRAM AND HUMAN NUTRITION PROGRAM  For many, the word “malnutrition” produces an image of a child in  a third-world country with a bloated belly, and skinny arms and  legs. However, this image alone is not an accurate representation of  the state of malnutrition. For example, someone who is 150 pounds  overweight can also be malnourished.  Malnutrition refers to one not receiving proper nutrition and does  not distinguish between the consequences of too many nutrients  or the lack of nutrients, both of which impair overall health.  Undernutrition is characterized by a lack of nutrients and  insufficient energy supply, whereas overnutrition is characterized  by excessive nutrient and energy intake. Overnutrition can result  in o

In [31]:
#now lets turn out list of dicitonaries into dataframes and get some stats
df=pd.DataFrame(pages_and_texts)
df.describe().round(2)

,Page_number,page_char_count,page_word_count,page_sentence_count,page_token_count,page_sentence_count_spacy
count,1208.00,1208.00,1208.00,1208.00,1208.00,1208.00
mean,562.50,1148.00,198.30,9.97,287.00,10.32
std,348.86,560.38,95.76,6.19,140.10,6.30
min,-41.00,0.00,1.00,1.00,0.00,0.00
25%,260.75,762.00,134.00,4.00,190.50,5.00
50%,562.50,1231.50,214.50,10.00,307.88,10.00
75%,864.25,1603.50,271.00,14.00,400.88,15.00
max,1166.00,2308.00,429.00,32.00,577.00,28.00


For our set of text, it looks like our raw sentence count (e.g. splitting on `". "`) is quite close to what spaCy came up with.

Now we've got our text split into sentences, how about we gorup those sentences?: